# Notebook 03 - Orchestration Demo

Demonstrates the full GiftConciergeAgent pipeline:

1. Product search with allergy-safe RAG
2. Preference update + memory persistence
3. Delivery check to various districts
4. Order history lookup

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.orchestrator import GiftConciergeAgent

agent = GiftConciergeAgent()
print('Agent ready:', agent.get_session_summary())

d:\Zuu Crew Agentic AI\Projects\Mini Project 03\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Loading embedding model: all-MiniLM-L6-v2…


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2493.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready
Agent ready: {'messages_in_context': 0, 'recipients_known': ['wife', 'mother', 'boss', 'daughter', 'friend_dinesh'], 'catalog_loaded': True}


In [2]:
# ── Demo 1: Product Search ──────────────────────────────
agent.reset_session()
response = agent.chat('I need a birthday gift for my wife Amaya, budget 5000 LKR')
print('Intent:', response['intent'])
print('Recipient:', response['recipient'])
print('Safety status:', response.get('safety_status', 'N/A'))
print('\n=== RESPONSE ===')
print(response['response'])

Intent: PRODUCT_SEARCH
Recipient: wife
Safety status: SAFE

=== RESPONSE ===
Hello! I'd be happy to recommend the best birthday gift for your wife Amaya within your LKR 5,000 budget.

Based on Amaya's preferences for sushi, gold jewelry, spa gift sets, dark chocolate, and red roses, as well as her allergies to peanuts, nuts, and shellfish, here are my top 3 gift recommendations:

1. Luxury Handmade Happy Birthday Milestone 60 Card (Tamil) - LKR 1,260
This beautifully crafted greeting card in Tamil is the perfect way to celebrate Amaya's special birthday milestone. The luxury handmade format and elegant design will make it a thoughtful and memorable gift. Plus, it's completely nut-free, dairy-free, and gluten-free, so you don't have to worry about any allergies.
Purchase link: https://www.kapruka.com/sri_lanka_search.jsp?searchWord=Happy%20Birthday%20Milestone%2060%20Card%20%E2%80%94%20Luxury%20Handmade%20%28Tamil%29

2. Luxury Handmade Happy Birthday Milestone 50 Card (English) - LKR 1

In [3]:
# ── Demo 2: Preference Update ───────────────────────────
response2 = agent.chat('Remember that my boss Mr Rajapakse is also allergic to shellfish now')
print('Intent:', response2['intent'])
print('Memory updated:', response2['memory_updated'])
print('\nResponse:', response2['response'])

# Verify the update persisted
boss_allergies = agent.memory.semantic.get_allergies('boss')
print('\nBoss allergies in memory:', boss_allergies)

Intent: PREFERENCE_UPDATE
Memory updated: True

Response: Thank you for providing the updated recipient information. I have noted that Mr. Rajapakse, your boss, is allergic to shellfish, and this will be strictly avoided in any gift recommendations. I will curate a selection of premium tea, corporate gift hampers, whisky accessories, and leather goods that align with his preferences and your budget of LKR 5,000 – 20,000. Please let me know if you have any other requirements, and I'll be happy to assist you further.

Boss allergies in memory: ['shellfish']


In [4]:
# ── Demo 3: Delivery Check ──────────────────────────────
response3 = agent.chat('Can you deliver a cake to Jaffna by Saturday?')
print('Intent:', response3['intent'])
print('\nResponse:', response3['response'])

Intent: DELIVERY_CHECK

Response: Thank you for your order with Kapruka Gift-Concierge! I'm happy to provide the delivery details for your perishable item to Jaffna.

Unfortunately, due to the perishable nature of the product, we are unable to deliver cakes or fresh flowers to the Jaffna district at this time. Our perishable item delivery is currently limited to the Colombo, Gampaha, and Kalutara districts only, to ensure the freshness of the items.

However, I can still assist you with a standard delivery to Jaffna. The estimated delivery time is 3 days, so your order would arrive approximately on Wednesday, April 08. The delivery fee for this order is LKR 900. Please note that we offer free delivery on orders over LKR 10,000.

I apologize for any inconvenience this may cause. If you have any other questions or need further assistance, please don't hesitate to let me know. I'm here to help make your gifting experience with Kapruka as smooth as possible.


In [5]:
# ── Demo 4: Products metadata ───────────────────────────
products = response.get('products_recommended', [])
print(f'Products tested against: {len(products)}')
meta = response.get('metadata', {})
print(f'Latency  : {meta.get("latency_ms", "?")} ms')
print(f'Model    : {meta.get("model_used", "?")}') 

Products tested against: 5
Latency  : 8759.0 ms
Model    : claude-3-haiku-20240307
